# tinyforge multi-GPU test (Kaggle, 2x T4)

Settings (right panel): **Accelerator = GPU T4 x2**, **Internet = On**. Then Run All (about 25-35 minutes).
It checks that NCCL works between the two GPUs, trains the same LoRA job on 1 GPU and on 2 GPUs, and compares final
loss, throughput, and whether the two replicas stayed identical. Paste the last cell's output back to the maintainer.

In [ ]:
import subprocess, json, time, os, pathlib, statistics
RESULTS = {}
def sh(cmd, name=None, timeout=3600, expect=0):
    t = time.time()
    try:
        p = subprocess.run(["bash", "-c", "set -o pipefail; " + cmd], capture_output=True, text=True, timeout=timeout)
        rc, out = p.returncode, (p.stdout + p.stderr)
    except subprocess.TimeoutExpired:
        rc, out = -9, "TIMEOUT"
    print(f"$ {cmd}\n{out[-2500:]}\n[exit {rc}, {time.time()-t:.0f}s]")
    if name: RESULTS[name] = (rc == expect)
    return rc
def check(name, ok, note=""):
    RESULTS[name] = bool(ok); print(("PASS " if ok else "FAIL ") + name, note)
sh("nvidia-smi --query-gpu=index,name,memory.total --format=csv; nvidia-smi topo -m | head -8; python --version")

In [ ]:
sh('pip install -q "tinyforge[finetune] @ git+https://github.com/rajratangm/tinyforge.git@main"', "install", timeout=1500)
sh("tinyforge doctor", "doctor")
import torch
n = torch.cuda.device_count(); print("GPUs visible:", n, [torch.cuda.get_device_name(i) for i in range(n)])
check("two_gpus_visible", n >= 2)

## 1. NCCL works between the GPUs (all-reduce and bandwidth)

In [ ]:
open("nccl_check.py", "w").write('''
import os, time, torch, torch.distributed as dist
dist.init_process_group("nccl"); r = dist.get_rank(); torch.cuda.set_device(int(os.environ["LOCAL_RANK"]))
x = torch.ones(1, device="cuda") * (r + 1); dist.all_reduce(x)
ok = x.item() == sum(range(1, dist.get_world_size() + 1))
big = torch.randn(64 * 1024 * 1024, device="cuda")  # 256 MB
for _ in range(2): dist.all_reduce(big)
torch.cuda.synchronize(); t = time.time()
for _ in range(5): dist.all_reduce(big)
torch.cuda.synchronize(); dt = (time.time() - t) / 5
if r == 0: print("RESULT allreduce_ok", ok, "algbw_GBps", round(0.256 / dt, 2), flush=True)
dist.destroy_process_group()
''')
sh("torchrun --standalone --nproc_per_node=2 nccl_check.py 2>&1 | grep -E 'RESULT|Error|error' | head", "nccl_allreduce")

## 2. Same LoRA job on 1 GPU, then on 2 GPUs

In [ ]:
sh("tinyforge ft data --source yahma/alpaca-cleaned --limit 1500 --out data/ft", "ft_data")
STEPS = 60
sh(f"tinyforge ft train --steps {STEPS} --data-dir data/ft --run-dir runs/g1 --gpus 1", "train_1gpu", timeout=2400)
sh(f"tinyforge ft train --steps {STEPS} --data-dir data/ft --run-dir runs/g2 --gpus 2", "train_2gpu", timeout=2400)

In [ ]:
def load(run):
    ev = [json.loads(l) for l in open(f"runs/{run}/metrics.jsonl")]
    steps = [e for e in ev if e["event"] == "step"]
    evals = [e for e in ev if e["event"] == "eval"]
    fin = [e for e in ev if e["event"] == "finished"]
    return steps, evals, (fin[-1] if fin else {})
s1, e1, f1 = load("g1"); s2, e2, f2 = load("g2")
tps1 = statistics.median(s["tok_per_s"] for s in s1[1:]) if len(s1) > 1 else 0
tps2 = statistics.median(s["tok_per_s"] for s in s2[1:]) if len(s2) > 1 else 0
print(f"1 GPU : median {tps1:,.0f} tok/s, peak mem {f1.get('peak_mem_gb', 0):.2f} GB, final val {e1[-1]['val_loss']:.4f}")
print(f"2 GPUs: median {tps2:,.0f} tok/s (all ranks), peak mem {f2.get('peak_mem_gb', 0):.2f} GB, final val {e2[-1]['val_loss']:.4f}")
print("speedup:", round(tps2 / tps1, 2) if tps1 else None, " world_size:", f2.get("world_size"),
      " replica divergence:", f2.get("ddp_max_param_divergence"))
w2 = [s.get("sync_s_per_step", 0) for s in s2[1:]]
sec2 = [b["t"] - a["t"] for a, b in zip(s2, s2[1:])]
print(f"2-GPU: median time waiting on the other GPU {statistics.median(w2 or [float("nan")]):.2f}s per step; median wall time per step {statistics.median(sec2 or [float("nan")]) / 5:.2f}s")
import math
check("val_losses_finite", all(math.isfinite(e["val_loss"]) for e in e1 + e2), "(no NaN from fp16 overflow)")
check("loss_decreased_1gpu", e1[-1]["val_loss"] < e1[0]["val_loss"])
check("loss_decreased_2gpu", e2[-1]["val_loss"] < e2[0]["val_loss"])
check("loss_matches_within_15pct", abs(e1[-1]["val_loss"] - e2[-1]["val_loss"]) / e1[-1]["val_loss"] < 0.15)
check("global_batch_is_2x", f2.get("examples_per_step_global") == 2 * f1.get("examples_per_step_global"),
      f"({f1.get('examples_per_step_global')} vs {f2.get('examples_per_step_global')} examples per step)")
check("replicas_identical", f2.get("ddp_max_param_divergence") == 0.0, f"({f2.get('ddp_max_param_divergence')})")
check("speedup_over_1.5x", tps1 > 0 and tps2 / tps1 > 1.5, f"({tps2 / tps1:.2f}x)" if tps1 else "")
check("adapter_saved_2gpu", pathlib.Path("runs/g2/best/adapter_model.safetensors").exists())

## 3. The job-spec worker with `gpus: 2` (what the Kubernetes agent would run)

In [ ]:
spec = '''apiVersion: tinyforge.dev/v1alpha1
kind: TrainingJob
metadata: {name: kaggle-2gpu}
spec:
  method: lora
  model: {base: HuggingFaceTB/SmolLM2-360M-Instruct}
  data: {source: yahma/alpaca-cleaned, format: alpaca, limit: 600, maxLen: 256}
  hyperparameters: {maxSteps: 30}
  resources: {gpus: 2}
  checkpoint: {everySteps: 15}
'''
open("job2.yaml", "w").write(spec)
sh("tinyforge worker run --spec job2.yaml --out job2_out 2>&1 | tail -6", "worker_2gpu", timeout=2400)
fin = [json.loads(l) for l in open("job2_out/metrics.jsonl") if '"finished"' in l] if os.path.exists("job2_out/metrics.jsonl") else []
check("worker_reports_world_size_2", bool(fin) and fin[-1].get("world_size") == 2)

## 4. Resume after interruption keeps both ranks in step

In [ ]:
sh("tinyforge ft train --steps 90 --data-dir data/ft --run-dir runs/g2 --gpus 2", "resume_2gpu", timeout=2400)
s2b, e2b, f2b = load("g2")
check("resumed_and_finished_90_steps", f2b.get("steps") == 90, f"(steps={f2b.get('steps')})")
check("resumed_replicas_identical", f2b.get("ddp_max_param_divergence") == 0.0)

## Summary

In [ ]:
for k, v in RESULTS.items(): print(("PASS " if v else "FAIL "), k)
bad = [k for k, v in RESULTS.items() if not v]
print()
print(f"{len(RESULTS)-len(bad)}/{len(RESULTS)} passed" + ("" if not bad else "; FAILED: " + ", ".join(bad)))